In [1]:
import os
import time
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input


In [2]:
DATASET_PATH = r"D:\Project Jupyter\MODELLING\Dataset_baruPR"

TRAIN_DIR = os.path.join(DATASET_PATH, "train")
VAL_DIR   = os.path.join(DATASET_PATH, "val")
TEST_DIR  = os.path.join(DATASET_PATH, "test")

BASE_MODEL_PATH =  r"D:\Project Jupyter\FINAL QUANTIZATION FIX\HASIL MODELBASE\mobilenetv3Large_jamur25COPYBARUFIXTF2PREPRO_tf"


IMG_SIZE = 224
BATCH_SIZE = 32


In [3]:
def preprocess_with_pad(image):
    image = tf.image.resize_with_pad(image, IMG_SIZE, IMG_SIZE)
    image = preprocess_input(image)
    return image


In [4]:
train_gen = ImageDataGenerator(
    preprocessing_function=preprocess_with_pad,
    rotation_range=10,
    zoom_range=0.1,
    brightness_range=[0.7, 1.3],
    horizontal_flip=True
)

val_gen = ImageDataGenerator(
    preprocessing_function=preprocess_with_pad
)

train_data = train_gen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=True
)

val_data = val_gen.flow_from_directory(
    VAL_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

test_gen = ImageDataGenerator(
    preprocessing_function=preprocess_with_pad
)

test_data = test_gen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)


attention_gen = ImageDataGenerator(
    preprocessing_function=preprocess_with_pad
)

attention_data = attention_gen.flow_from_directory(
    TRAIN_DIR,                 # pakai TRAIN set
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=8,              # ⬅️ kecil supaya tidak OOM
    class_mode="categorical",
    shuffle=False
)


class_names = list(test_data.class_indices.keys())
print("Class:", class_names)

Found 2256 images belonging to 2 classes.
Found 282 images belonging to 2 classes.
Found 232 images belonging to 2 classes.
Found 2256 images belonging to 2 classes.
Class: ['edible', 'poisonous']


In [9]:
model = tf.keras.models.load_model(
    BASE_MODEL_PATH,
    compile=False
)

model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 MobilenetV3large (Functiona  (None, 7, 7, 960)        2996352   
 l)                                                              
                                                                 
 global_average_pooling2d (G  (None, 960)              0         
 lobalAveragePooling2D)                                          
                                                                 
 dense (Dense)               (None, 128)               123008    
                                                                 
 dropout (Dropout)           (None, 128)               0         
                                                                 
 dense_1 (Dense)             (None, 2)                 258       
                                                                 
Total params: 3,119,618
Trainable params: 123,266
Non-tr

In [10]:
backbone = model.layers[0]     # MobileNetV3Large
classifier = model.layers[1:]  # GAP + Dense + Dropout + Dense


In [11]:
def extract_se_attention(backbone, dataset, max_batches=20):
    """
    return: list of attention vectors shape (960,)
    """
    se_layers = [
        l for l in backbone.layers
        if "squeeze_excite" in l.name and isinstance(l, tf.keras.layers.Multiply)
    ]

    assert len(se_layers) > 0, "SE layer tidak ditemukan"

    # ambil SE TERAKHIR (output 960)
    se_layer = se_layers[-1]

    att_model = tf.keras.Model(
        backbone.input,
        se_layer.output
    )

    attentions = []

    for i, (x, _) in enumerate(dataset):
        if i >= max_batches:
            break

        se_out = att_model(x, training=False)
        # (B, 1, 1, 960)
        att = tf.reduce_mean(se_out, axis=[0, 1, 2])  # (960,)
        attentions.append(att.numpy())

    return attentions


In [12]:
se_attentions = extract_se_attention(backbone, attention_data)

final_attention = np.mean(
    np.stack(se_attentions, axis=0),
    axis=0
)

print(final_attention.shape)
# HARUS (960,)


(960,)


In [13]:
def build_channel_mask(attention, prune_ratio=0.3):
    C = attention.shape[0]
    keep = int(C * (1 - prune_ratio))

    threshold = np.sort(attention)[-keep]
    mask = (attention >= threshold).astype(np.float32)

    print("Keep channels:", mask.sum(), "/", C)
    return mask


In [14]:
channel_mask = build_channel_mask(final_attention, prune_ratio=0.3)

Keep channels: 672.0 / 960


In [15]:
class ChannelMask(tf.keras.layers.Layer):
    def __init__(self, mask, **kwargs):
        super().__init__(trainable=False, **kwargs)
        self.mask = tf.reshape(
            tf.constant(mask, dtype=tf.float32),
            [1, 1, 1, -1]
        )

    def call(self, x):
        return x * self.mask


In [16]:
inputs = tf.keras.Input(shape=(224,224,3))

x = backbone(inputs, training=False)
x = ChannelMask(channel_mask, name="channel_prune_mask")(x)

for layer in classifier:
    x = layer(x)

pruned_model = Model(inputs, x)
pruned_model.summary()


Model: "model_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 224, 224, 3)]     0         
                                                                 
 MobilenetV3large (Functiona  (None, 7, 7, 960)        2996352   
 l)                                                              
                                                                 
 channel_prune_mask (Channel  (None, 7, 7, 960)        0         
 Mask)                                                           
                                                                 
 global_average_pooling2d (G  (None, 960)              0         
 lobalAveragePooling2D)                                          
                                                                 
 dense (Dense)               (None, 128)               123008    
                                                           

In [17]:
import time

EPOCHS_FINE_TUNE = 10


pruned_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True,
    verbose=1
)


start_time = time.time()


history = pruned_model.fit(
    train_data,
    validation_data=val_data,
    epochs=EPOCHS_FINE_TUNE,
    callbacks=[early_stop],
    verbose=1
)

end_time = time.time()

training_time = end_time - start_time

print(f"\nTotal Training Time : {training_time:.2f} seconds")
print(f"Total Training Time : {training_time/60:.2f} minutes")
print(f"Avg Time / Epoch    : {training_time/EPOCHS_FINE_TUNE:.2f} seconds")

Epoch 1/10
71/71 [==============================] - 360s 5s/step - loss: 0.2609 - accuracy: 0.8932 - val_loss: 0.2647 - val_accuracy: 0.9078
Epoch 2/10
71/71 [==============================] - 264s 4s/step - loss: 0.2485 - accuracy: 0.9051 - val_loss: 0.2542 - val_accuracy: 0.9220
Epoch 3/10
71/71 [==============================] - 359s 5s/step - loss: 0.2277 - accuracy: 0.9113 - val_loss: 0.2470 - val_accuracy: 0.9326
Epoch 4/10
71/71 [==============================] - 256s 4s/step - loss: 0.2143 - accuracy: 0.9158 - val_loss: 0.2403 - val_accuracy: 0.9362
Epoch 5/10
71/71 [==============================] - 255s 4s/step - loss: 0.1999 - accuracy: 0.9291 - val_loss: 0.2354 - val_accuracy: 0.9433
Epoch 6/10
71/71 [==============================] - 253s 4s/step - loss: 0.1929 - accuracy: 0.9313 - val_loss: 0.2235 - val_accuracy: 0.9362
Epoch 7/10
71/71 [==============================] - 251s 4s/step - loss: 0.1884 - accuracy: 0.9353 - val_loss: 0.2198 - val_accuracy: 0.9433
Epoch 8/10
71

In [18]:
PRUNED_MODEL_PATH = r"D:\mobilenetv3_PRUNED_FINAL30AGP"

pruned_model.save(
    PRUNED_MODEL_PATH,
    include_optimizer=False
)


INFO:tensorflow:Assets written to: D:\mobilenetv3_se_attention_pruned_FINAL30AGP10BANGET\assets


INFO:tensorflow:Assets written to: D:\mobilenetv3_se_attention_pruned_FINAL30AGP10BANGET\assets


In [19]:
import os

def get_model_size_mb(model_path):
    total_size = 0
    for dirpath, dirnames, filenames in os.walk(model_path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            total_size += os.path.getsize(fp)
    return total_size / (1024 * 1024)


In [20]:
base_model_size = get_model_size_mb(BASE_MODEL_PATH)
pruned_model_size = get_model_size_mb(PRUNED_MODEL_PATH)

print(f"Ukuran Baseline Model : {base_model_size:.2f} MB")
print(f"Ukuran Pruned Model   : {pruned_model_size:.2f} MB")

compression_ratio = base_model_size / pruned_model_size
size_reduction = (1 - (pruned_model_size / base_model_size)) * 100

print(f"Compression Ratio : {compression_ratio:.2f}x")
print(f"Size Reduction    : {size_reduction:.2f}%")


Ukuran Baseline Model : 17.87 MB
Ukuran Pruned Model   : 16.82 MB
Compression Ratio : 1.06x
Size Reduction    : 5.83%


In [30]:
from sklearn.metrics import classification_report, confusion_matrix


In [31]:
test_data.reset()


In [32]:
base_model = tf.keras.models.load_model(
    PRUNED_MODEL_PATH,
    compile=False
)

predictions = base_model.predict(test_data, verbose=1)
y_pred = np.argmax(predictions, axis=1)
y_true = test_data.classes


8/8 [==============================] - 25s 3s/step


In [33]:
print("\nClassification Report:")
print(classification_report(
    y_true,
    y_pred,
    target_names=class_names
))



Classification Report:
              precision    recall  f1-score   support

      edible       0.96      0.93      0.95       120
   poisonous       0.93      0.96      0.94       112

    accuracy                           0.94       232
   macro avg       0.94      0.94      0.94       232
weighted avg       0.94      0.94      0.94       232



In [34]:
print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))



Confusion Matrix:
[[112   8]
 [  5 107]]


In [35]:
import time
import numpy as np

def benchmark_inference(model, data_gen, warmup=3, runs=10):
    """
    model      : final pruned model (tanpa pruning wrapper)
    data_gen   : test_data (shuffle=False)
    """

    # Ambil 1 batch
    x_batch, _ = next(data_gen)

    # Warm-up (penting)
    for _ in range(warmup):
        _ = model.predict(x_batch, verbose=0)

    times = []

    for _ in range(runs):
        start = time.time()
        _ = model.predict(x_batch, verbose=0)
        end = time.time()
        times.append(end - start)

    avg_time = np.mean(times)
    per_image_time = avg_time / x_batch.shape[0]

    print("\nInference Benchmark (CPU)")
    print(f"Batch size         : {x_batch.shape[0]}")
    print(f"Avg batch time     : {avg_time:.4f} seconds")
    print(f"Avg per image time : {per_image_time:.6f} seconds")

    return avg_time, per_image_time


In [36]:
benchmark_inference(base_model, test_data)



Inference Benchmark (CPU)
Batch size         : 32
Avg batch time     : 1.4448 seconds
Avg per image time : 0.045151 seconds


(1.4448453426361083, 0.045151416957378385)